# 離線小恐龍自動操控

先切到 Chrome 離線小恐龍遊戲，再執行下方 cell；程式先等待 5 秒，再以 PyAutoGUI 按空白鍵開始。執行中按 **Esc** 停止。

## 尺寸與遊戲參數（常見 Chromium 版本，畫面縮放會影響截圖像素）

| 項目 | 寬 × 高 / 數值 |
|---|---:|
| 恐龍 | 依提供範圍 x=90–235，寬 145 px；高度按原比例估約 155 px |
| 小仙人掌 | 約 56 × 115 px；雙／三叢約 112–132／168–204 px |
| 大仙人掌 | 約 82 × 165 px；雙／三叢約 165–191／247–290 px |
| 翼龍 | 約 151 × 132 px；偵測線依新縮放設定 |
| 初始速度／加速度／最大速度 | 約 19.8／0.0033／42.8 px 每遊戲影格 |
| 跳躍模型 | 約 30／36 影格；距離依縮放後速度估算 |
| 障礙物色彩 | 亮色主題約 RGB(83,83,83)；暗色主題約 RGB(247,247,247) |

座標依恐龍 x=90–235（寬 145 px）相對原寬 44 px 等比例換算，比例約 3.30；地面設為 y=805。障礙偵測 x 範圍為 235–765；下方偵測線 y=700、745，上方線 y=625。大仙人掌高度取樣 y=640、656、672，恐龍落地檢查腳部 y=795–804。每隔 33 px 取樣，下方兩線錯開約 16 px，並以像素寬高分類仙人掌／翼龍。起跳門檻是障礙物左緣的絕對螢幕座標：初始為 x=235–350；速度增加或需等待落地／放開按鍵時，只向右提高上界。障礙進入偵測範圍即保留待跳指令；短暫漏偵測時依最後位置與速度推估。起跳時按一次空白鍵，大型仙人掌按住 0.5 秒，其餘按住 0.01 秒。輸出：障礙首次進入起跳範圍 `1`、即時跳躍 `2,秒數`、等待後跳躍 `3,秒數`、進入起跳範圍但未跳 `4`；另輸出空白鍵按下至放開間隔及落地至按鍵間隔。最近障礙連續 0.75 秒位置不變則輸出 `break` 並停止。

若截圖依賴載入失敗，請在終端機執行 `conda run -n iem_python python -m pip install --upgrade Pillow pyscreeze pyautogui`，再重新啟動 VS Code 的 Python kernel。


In [ ]:
import ctypes
import importlib
import time
try:
    import pyscreeze
    from PIL import Image
except ImportError as error:
    raise RuntimeError("缺少截圖套件，請安裝 Pillow、pyscreeze：conda run -n iem_python python -m pip install --upgrade Pillow pyscreeze pyautogui") from error
import pyautogui as pyg
pyg = importlib.reload(pyg)  # 重新載入可能曾因缺少 Pillow 而停用截圖的模組

# 以恐龍 x=90–235、地面 y=805 為基準；其他尺寸依寬度比例等比例推算
DINO_FRONT, DINO_W = 235, 145
DINO_X = DINO_FRONT - DINO_W
DINO_SCALE = DINO_W / 44
def scaled(value):
    return max(1, round(value * DINO_SCALE))
X_MIN, X_MAX = DINO_FRONT, 765
GROUND_Y = 805
LOW_Y, LOW_Y_EXTRA, MID_Y = 700, 745, 625
SAMPLE_STEP, SCREEN_Y0 = scaled(10), MID_Y
FPS, START_SPEED, ACCEL, MAX_SPEED = 60, 6.0 * DINO_SCALE, 0.001 * DINO_SCALE, 13.0 * DINO_SCALE
SHORT_JUMP_FRAMES, LONG_JUMP_FRAMES = 30, 36
TAKEOFF_X_MIN, TAKEOFF_X_BASE_MAX = DINO_FRONT, 350  # 絕對螢幕 x 座標
JUMP_SPEED_FACTOR = 2.0 * DINO_SCALE
JUMP_COMMAND_MARGIN = 0.05       # 預留截圖／鍵盤指令處理時間（秒）
SHORT_HOLD, LONG_HOLD = 0.01, 0.5
LANDING_LEAD = 0.05              # 估計落地前提前送出指令，補償輸出延遲
ZERO_SPEED_TIMEOUT = 0.75        # 最近障礙物靜止多久後判定速度為 0
pyg.PAUSE = 0


def esc_pressed():
    return hasattr(ctypes, "windll") and bool(ctypes.windll.user32.GetAsyncKeyState(0x1B) & 0x8000)


def pixel(im, x, y):
    return im.getpixel((x, y - SCREEN_Y0))


def runs(im, y, offset=0, min_points=1):
    """在指定水平線找障礙物像素群，容許少量抗鋸齒間隙。"""
    bg = pixel(im, 0, y)
    target = (247, 247, 247) if sum(bg[:3]) < 300 else (83, 83, 83)
    start = max(X_MIN, DINO_FRONT) + offset
    xs = [x for x in range(start, min(X_MAX, im.width - 1) + 1, SAMPLE_STEP)
          if not DINO_X <= x < DINO_FRONT and
          max(abs(pixel(im, x, y)[c] - target[c]) for c in range(3)) < 65]
    groups = []
    for x in xs:
        if not groups or x - groups[-1][-1] > SAMPLE_STEP + scaled(8):
            groups.append([x])
        else:
            groups[-1].append(x)
    return [(g[0], g[-1]) for g in groups if len(g) >= min_points]


def merge_runs(*lines):
    merged = []
    for left, right in sorted(run for line in lines for run in line):
        if merged and left <= merged[-1][1] + scaled(8):
            merged[-1] = (merged[-1][0], max(merged[-1][1], right))
        else:
            merged.append((left, right))
    return merged


def classify_lower(im, left, right):
    width = right - left + SAMPLE_STEP // 2 + 1  # 下方兩線錯開取樣後的寬度估值
    if scaled(42) <= width <= scaled(49):
        return "pterodactyl", False
    sample_ys = range(GROUND_Y - scaled(50), GROUND_Y - scaled(39) + 1, scaled(5))
    targets = {y: ((247, 247, 247) if sum(pixel(im, 0, y)[:3]) < 300 else (83, 83, 83))
               for y in sample_ys}
    tall = any(
        max(abs(pixel(im, x, y)[c] - targets[y][c]) for c in range(3)) < 65
        for x in range(left, right + 1, SAMPLE_STEP)
        for y in sample_ys
    )
    if tall:
        group = "single" if width <= scaled(34) else "double" if width <= scaled(62) else "triple"
        return f"large_cactus_{group}", True
    group = "single" if width <= scaled(22) else "double" if width <= scaled(45) else "triple"
    return f"small_cactus_{group}", False


def obstacles(im, lower, upper):
    found = [(l, r, *classify_lower(im, l, r)) for l, r in lower]
    for l, r in upper:
        if not any(not (r < a - scaled(5) or l > b + scaled(5)) for a, b in lower):
            found.append((l, r, "pterodactyl", False))
    return sorted(found, key=lambda item: item[0])


def is_grounded(im):
    bg = pixel(im, 0, GROUND_Y - 1)
    target = (247, 247, 247) if sum(bg[:3]) < 300 else (83, 83, 83)
    pixels = sum(
        max(abs(pixel(im, x, y)[c] - target[c]) for c in range(3)) < 65
        for x in range(DINO_X, DINO_FRONT)
        for y in range(GROUND_Y - scaled(3), GROUND_Y)
    )
    return pixels >= scaled(2)


def travel(frames, speed):
    """按目前速度、加速度與最大速度估算指定影格的移動距離。"""
    accelerating = min(frames, max(0, (MAX_SPEED - speed) / ACCEL))
    return speed * accelerating + 0.5 * ACCEL * accelerating**2 + MAX_SPEED * (frames - accelerating)


print("5 秒後開始；請先切換到遊戲視窗。按 Esc 可取消。")
for _ in range(50):
    if esc_pressed():
        break
    time.sleep(0.1)
else:
    pyg.press("space")
    speed = START_SPEED
    previous_x, previous_time = None, time.perf_counter()
    stationary_since = None
    jumping, airborne_seen = False, False
    jump_start, jump_target, jump_speed = 0, None, START_SPEED
    estimated_landing_at = None
    space_release_at, space_hold = None, 0
    space_down_at = None
    last_landed_at = None
    pending_jump = False
    pending_candidate = None
    detected_at = None
    tracked_obstacle_x, tracked_obstacle_time = None, None
    obstacle_last_seen = None
    pending_waited_for_air = False
    blocked_reported = False
    takeoff_range_reported = False
    DIAGNOSTIC_SECONDS = 50
    diagnostic_start = time.perf_counter()
    diagnostic_frames = 0
    loop_total = loop_max = 0.0
    screenshot_total = screenshot_max = 0.0
    release_count = 0
    release_late_total = release_late_max = 0.0
    diagnostic_reported = False

    def report_timing():
        elapsed = time.perf_counter() - diagnostic_start
        avg_loop = loop_total / diagnostic_frames * 1000 if diagnostic_frames else 0
        avg_shot = screenshot_total / diagnostic_frames * 1000 if diagnostic_frames else 0
        avg_late = release_late_total / release_count * 1000 if release_count else 0
        release_stats = (f"{avg_late:.2f}/{release_late_max * 1000:.2f} ms (平均/最大)"
                         if release_count else "無空白鍵放開樣本")
        print(f"[計時診斷 {elapsed:.1f}s] 迴圈平均/最大 {avg_loop:.2f}/{loop_max * 1000:.2f} ms；"
              f"截圖平均/最大 {avg_shot:.2f}/{screenshot_max * 1000:.2f} ms；"
              f"空白鍵放開延遲 {release_stats}")

    try:
        while not esc_pressed():
            loop_started = time.perf_counter()
            now = loop_started
            if space_release_at is not None and now >= space_release_at:
                release_started = time.perf_counter()
                release_late = max(0.0, release_started - space_release_at)
                release_count += 1
                release_late_total += release_late
                release_late_max = max(release_late_max, release_late)
                pyg.keyUp("space")
                if space_down_at is not None:
                    print(f"空白鍵按下至放開,{release_started - space_down_at:.3f}s")
                    space_down_at = None
                space_release_at = None
            screenshot_started = time.perf_counter()
            im = pyg.screenshot(region=(0, SCREEN_Y0, X_MAX + 1, GROUND_Y - SCREEN_Y0 + 1))
            screenshot_elapsed = time.perf_counter() - screenshot_started
            screenshot_total += screenshot_elapsed
            screenshot_max = max(screenshot_max, screenshot_elapsed)
            on_ground = is_grounded(im)
            if not on_ground:
                jumping, airborne_seen = True, True
            elif jumping and airborne_seen:
                jumping, airborne_seen = False, False
                last_landed_at = time.perf_counter()
                estimated_landing_at = None
            jump_ready = (on_ground and not jumping) or (
                jumping and airborne_seen and estimated_landing_at is not None
                and now >= estimated_landing_at - LANDING_LEAD
            )
            lower = merge_runs(runs(im, LOW_Y), runs(im, LOW_Y_EXTRA, offset=SAMPLE_STEP // 2))
            upper = runs(im, MID_Y, min_points=2)

            # 追蹤最近障礙物移動速度；長時間靜止視為遊戲速度歸零
            nearest = min((l for l, r in lower + upper if r >= DINO_FRONT), default=None)
            dt = now - previous_time
            if nearest is None:
                previous_x, previous_time, stationary_since = None, now, None
            else:
                if previous_x is None:
                    stationary_since = now
                elif dt > 0:
                    dx = previous_x - nearest
                    if nearest <= DINO_FRONT + scaled(15):
                        # 障礙已貼近恐龍時，左緣會被 ROI 截斷；不拿截斷座標判定靜止
                        stationary_since = None
                    elif 0 < dx < MAX_SPEED * FPS * dt * 1.8:
                        measured = dx / (dt * FPS)
                        speed = min(MAX_SPEED, 0.75 * speed + 0.25 * measured)
                        stationary_since = None
                    elif dx == 0:
                        if stationary_since is None:
                            stationary_since = previous_time
                    else:
                        stationary_since = now
                previous_x, previous_time = nearest, now
                if stationary_since is not None and now - stationary_since >= ZERO_SPEED_TIMEOUT:
                    speed = 0.0
            if speed <= 0:
                print("break")
                diagnostic_reported = True
                break

            # 合併兩條線上的障礙物，依類型與速度設定待跳指令
            found = obstacles(im, lower, upper)
            active_x = None
            if jump_target is not None:
                active_x = jump_target - travel(max(0, (now - jump_start) * FPS), jump_speed)
                if active_x < DINO_FRONT:
                    jump_target, active_x = None, None
            candidate = next((item for item in found if item[0] >= TAKEOFF_X_MIN and
                              (active_x is None or abs(item[0] - active_x) > max(scaled(20), item[1] - item[0] + scaled(10)))), None)
            # 偵測範圍內即保留指令；起跳距離是另一個獨立門檻
            available_at = now
            if not jump_ready:
                if estimated_landing_at is not None:
                    available_at = max(available_at, estimated_landing_at - LANDING_LEAD)
                else:
                    # 首次起始跳沒有估計落地時間，以最長跳躍時間保守估算
                    available_at = now + LONG_JUMP_FRAMES / FPS
            if space_release_at is not None:
                available_at = max(available_at, space_release_at)
            wait_to_jump = max(0.0, available_at - now)
            if wait_to_jump > 0:
                wait_to_jump += JUMP_COMMAND_MARGIN
            takeoff_x_max = min(X_MAX, TAKEOFF_X_BASE_MAX
                               + max(0, speed - START_SPEED) * JUMP_SPEED_FACTOR
                               + max(0.0, speed) * FPS * wait_to_jump)
            action_candidate = candidate
            if candidate is not None:
                same_obstacle = False
                if tracked_obstacle_x is not None and tracked_obstacle_time is not None:
                    expected_x = tracked_obstacle_x - speed * FPS * max(0, now - tracked_obstacle_time)
                    tolerance = max(scaled(20), candidate[1] - candidate[0] + scaled(10))
                    same_obstacle = abs(candidate[0] - expected_x) <= tolerance
                if not same_obstacle:
                    detected_at = now
                    pending_waited_for_air, blocked_reported = False, False
                    takeoff_range_reported = False
                    pending_candidate = candidate
                tracked_obstacle_x, tracked_obstacle_time = candidate[0], now
                obstacle_last_seen = now
                pending_jump = True
                pending_candidate = candidate
            elif pending_jump and pending_candidate is not None and tracked_obstacle_x is not None:
                # 線上取樣偶爾漏掉同一障礙時，不清除待跳；用最近一次位置外推
                predicted_x = tracked_obstacle_x - speed * FPS * max(0, now - tracked_obstacle_time)
                if predicted_x < TAKEOFF_X_MIN:
                    pending_jump = False
                    pending_candidate = None
                    detected_at = None
                    tracked_obstacle_x, tracked_obstacle_time, obstacle_last_seen = None, None, None
                    pending_waited_for_air, blocked_reported = False, False
                    takeoff_range_reported = False
                else:
                    width = pending_candidate[1] - pending_candidate[0]
                    action_candidate = (round(predicted_x), round(predicted_x + width),
                                        pending_candidate[2], pending_candidate[3])
            in_takeoff_range = (action_candidate is not None and
                               TAKEOFF_X_MIN <= action_candidate[0] <= takeoff_x_max)

            if in_takeoff_range and not takeoff_range_reported:
                print(1)
                takeoff_range_reported = True

            if pending_jump and not jump_ready:
                pending_waited_for_air = True
            can_jump_now = jump_ready and space_release_at is None
            if pending_jump and in_takeoff_range and not can_jump_now and not blocked_reported:
                print(4)
                blocked_reported = True
            # 待跳指令等到起跳距離且可跳時立刻執行
            if pending_jump and in_takeoff_range and can_jump_now:
                jump_target = action_candidate[0]
                if on_ground and not jumping:
                    landing_reference_at, landing_reference_type = last_landed_at, "實測"
                else:
                    landing_reference_at, landing_reference_type = estimated_landing_at, "預估"
                press_started = time.perf_counter()
                jump_speed, jump_start = speed, press_started
                jumping, airborne_seen = True, False
                long_jump = action_candidate[2].startswith("large_cactus")
                jump_frames = LONG_JUMP_FRAMES if long_jump else SHORT_JUMP_FRAMES
                estimated_landing_at = jump_start + LANDING_LEAD + jump_frames / FPS
                space_hold = LONG_HOLD if long_jump else SHORT_HOLD
                pending_jump = False
                pending_candidate = None
                output_code = 3 if pending_waited_for_air else 2
                detection_elapsed = press_started - detected_at if detected_at is not None else 0.0
                pyg.keyDown("space")
                space_down_at = press_started
                print(f"{output_code},{detection_elapsed:.3f}")
                if landing_reference_at is None:
                    print("落地至空白鍵按下,無落地樣本")
                else:
                    landing_interval = press_started - landing_reference_at
                    print(f"落地至空白鍵按下({landing_reference_type}),{landing_interval:+.3f}s")
                pending_waited_for_air, blocked_reported = False, False
                space_release_at = press_started + space_hold
            time.sleep(0.01)
            loop_elapsed = time.perf_counter() - loop_started
            loop_total += loop_elapsed
            loop_max = max(loop_max, loop_elapsed)
            diagnostic_frames += 1
            if not diagnostic_reported and time.perf_counter() - diagnostic_start >= DIAGNOSTIC_SECONDS:
                report_timing()
                diagnostic_reported = True
    finally:
        if not diagnostic_reported:
            report_timing()
        if space_down_at is not None:
            final_release_at = time.perf_counter()
            try:
                pyg.keyUp("space")
            finally:
                print(f"空白鍵按下至放開,{final_release_at - space_down_at:.3f}s")
                space_down_at = None
        for key in ("space",):
            try:
                pyg.keyUp(key)
            except Exception:
                pass


5 秒後開始；請先切換到遊戲視窗。按 Esc 可取消。
1
2,0.926
落地至空白鍵按下,無落地樣本
空白鍵按下至放開,0.012s
4
break


In [ ]:
import ctypes
import time

import pyautogui as pyg


def esc_pressed():
    return bool(ctypes.windll.user32.GetAsyncKeyState(0x1B) & 0x8000)


pyg.PAUSE = 0
DINO_FRONT, DINO_W, SCREEN_RIGHT = 235, 145, 765
DINO_X = DINO_FRONT - DINO_W
TOP, GROUND, STEP = 625, 805, 8
FPS, START_SPEED, ACCEL, MAX_SPEED = 60, 19.8, 0.0033, 42.9
LEAD_FRAMES, JUMP_FRAMES = 18, 36

def is_target(im, x, y, target):
    color = im.getpixel((x, y - TOP))
    return max(abs(color[c] - target[c]) for c in range(3)) < 65

def line_runs(im, y, target):
    xs = [x for x in range(DINO_FRONT + 10, SCREEN_RIGHT + 1, STEP)
          if is_target(im, x, y, target)]
    groups = []
    for x in xs:
        if not groups or x - groups[-1][-1] > STEP * 2:
            groups.append([x])
        else:
            groups[-1].append(x)
    return [group[0] for group in groups if len(group) >= 2]

def grounded(im, target):
    return sum(is_target(im, x, y, target)
               for x in range(DINO_X, DINO_FRONT, 5)
               for y in range(GROUND - 10, GROUND)) >= 5

def obstacle_positions(im, target):
    ys = range(TOP, 746, 10)  # 加密垂直取樣，涵蓋中高度翼龍
    hits = sorted((x, y) for y in ys for x in line_runs(im, y, target))
    groups = []
    for hit in hits:
        if not groups or hit[0] - groups[-1][-1][0] > STEP * 3:
            groups.append([hit])
        else:
            groups[-1].append(hit)
    return [min(x for x, _ in group) for group in groups
            if len({y for _, y in group}) >= 2]

def travel(frames, speed):
    accelerating = min(frames, max(0, (MAX_SPEED - speed) / ACCEL))
    return (speed * accelerating + 0.5 * ACCEL * accelerating**2
            + MAX_SPEED * (frames - accelerating))

print("3 秒後開始；請先切到遊戲，按 Esc 停止。")
for _ in range(30):
    if esc_pressed():
        break
    time.sleep(0.1)
else:
    region = (0, TOP, SCREEN_RIGHT + 1, GROUND - TOP + 1)
    pyg.press("space")
    started = previous_at = time.perf_counter()
    speed = START_SPEED
    previous_x = tracked_x = tracked_at = None
    pending = handled = landing_pending = False
    jump_started = jump_obstacle_x = None
    jump_speed = START_SPEED
    misses = 0
    while not esc_pressed():
        now = time.perf_counter()
        im = pyg.screenshot(region=region)
        bg = im.getpixel((0, GROUND - TOP - 1))
        target = (247, 247, 247) if sum(bg[:3]) < 300 else (83, 83, 83)
        positions = obstacle_positions(im, target)
        obstacle_x = positions[0] if positions else None
        on_ground = grounded(im, target)
        frames = (now - started) * FPS
        speed = max(speed, min(START_SPEED + ACCEL * frames, MAX_SPEED))
        detect_x = min(DINO_FRONT + round(speed * LEAD_FRAMES), SCREEN_RIGHT)
        jumped = False

        if obstacle_x is not None:
            misses = 0
            dt = now - previous_at
            if previous_x is not None and dt > 0:
                measured = (previous_x - obstacle_x) / (dt * FPS)
                if 0 < measured <= MAX_SPEED * 1.5:
                    speed = min(MAX_SPEED, 0.7 * speed + 0.3 * measured)
            previous_x, previous_at = obstacle_x, now

            same = False
            if tracked_x is not None:
                elapsed = now - tracked_at
                expected = tracked_x - speed * FPS * elapsed
                same = abs(obstacle_x - expected) < max(70, speed * FPS * elapsed * 1.5)
            if not same:
                pending = handled = False
            tracked_x, tracked_at = obstacle_x, now

            if obstacle_x <= detect_x:
                pending = True
            if pending and on_ground and not handled and obstacle_x > DINO_FRONT + 20:
                pyg.press("space")
                print(f"跳躍：障礙 x={obstacle_x}，偵測線={detect_x}，速度={speed:.1f}px/影格")
                pending, handled = False, True
                jump_started, jump_obstacle_x, jump_speed = now, obstacle_x, speed
                landing_pending, jumped = False, True
        else:
            misses += 1
            if misses >= 3:
                previous_x = tracked_x = tracked_at = None
                pending = handled = False

        if not on_ground and jump_started is not None:
            elapsed_frames = (now - jump_started) * FPS
            passed_x = jump_obstacle_x - travel(elapsed_frames, jump_speed)
            next_obstacles = [x for x in positions
                              if x > max(DINO_FRONT + 20, passed_x + 60)]
            remaining = max(0, JUMP_FRAMES - elapsed_frames)
            if next_obstacles and remaining > 0:
                landing_x = next_obstacles[0] - travel(remaining, speed)
                landing_speed = min(MAX_SPEED, speed + ACCEL * remaining)
                landing_line = min(DINO_FRONT + round(landing_speed * LEAD_FRAMES), SCREEN_RIGHT)
                if DINO_FRONT + 20 < landing_x <= landing_line and not landing_pending:
                    landing_pending = True
                    print(f"落點預警：依目前速度，落地時障礙約在 x={landing_x:.0f}")

        if landing_pending and on_ground and not jumped:
            passed_x = (jump_obstacle_x - travel((now - jump_started) * FPS, jump_speed)
                        if jump_started is not None else DINO_FRONT)
            next_obstacles = [x for x in positions
                              if x > max(DINO_FRONT + 20, passed_x + 60)]
            if next_obstacles and DINO_FRONT + 20 < next_obstacles[0] <= detect_x:
                landing_obstacle = next_obstacles[0]
                pyg.press("space")
                print(f"落地連跳：障礙 x={landing_obstacle}，速度={speed:.1f}px/影格")
                jump_started, jump_obstacle_x, jump_speed = now, landing_obstacle, speed
                tracked_x, tracked_at, handled = landing_obstacle, now, True
                pending, jumped = False, True
            landing_pending = False

        time.sleep(0.005)
print("已停止。")
